# 📊 Phase 1 — Part 4: Split Dataset into Train / Validation / Test

---

## What this notebook does
Takes your **manually verified** CSVs and splits them:
- **70% Train** — model learns from this
- **15% Validation** — used during training to check progress
- **15% Test** — used ONLY at the end to report final WER

## Prerequisites
✅ You must have completed **manual correction** of the CSVs from Part 3.
The `verified` column must be `YES` for rows you want to use.

## What gets saved
| File | Purpose |
|------|---------|
| `dataset/dhaka_train.csv` | Training data |
| `dataset/dhaka_val.csv` | Validation data |
| `dataset/dhaka_test.csv` | Test data (don't touch until evaluation) |
| Same for `chittagong_*.csv` | Same split for Chittagong |
| `scripts/split_dataset.py` | Script backup |

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

In [ ]:
!pip install -q scikit-learn pandas
print('✅ Dependencies ready')

## Step 1: Check Verification Status
Let's see how many clips you've verified.

In [ ]:
import pandas as pd

for dialect in ['dhaka', 'chittagong']:
    csv_path = os.path.join(BASE, f'dataset/{dialect}_raw.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        verified = df[df.verified == 'YES']
        print(f'📋 {dialect}_raw.csv:')
        print(f'   Total rows:    {len(df)}')
        print(f'   Verified (YES): {len(verified)}')
        print(f'   Not verified:   {len(df) - len(verified)}')
        if len(verified) < 100:
            print(f'   ⚠️ WARNING: {len(verified)} verified clips is very low. Aim for 300+.')
        print()
    else:
        print(f'⚠️ {dialect}_raw.csv not found. Run Part 3 first.\n')

## Step 2: Split Into Train / Validation / Test

### Split ratio: 70% / 15% / 15%
- `random_state=42` ensures the split is **reproducible** — same result every time
- Only uses rows where `verified == YES`

In [ ]:
# ════════════════════════════════════════════════════════════════
# SPLIT DATASET — 70% train / 15% val / 15% test
# ════════════════════════════════════════════════════════════════

from sklearn.model_selection import train_test_split

for dialect in ['dhaka', 'chittagong']:
    csv_path = os.path.join(BASE, f'dataset/{dialect}_raw.csv')
    if not os.path.exists(csv_path):
        print(f'⚠️ {dialect}_raw.csv not found. Skipping.')
        continue
    
    df = pd.read_csv(csv_path)
    df = df[df.verified == 'YES']  # Only use verified rows
    
    if len(df) < 10:
        print(f'⚠️ {dialect}: Only {len(df)} verified clips. Need at least 10. Skipping.')
        continue
    
    # Split: 70% train, 30% temp
    train, temp = train_test_split(df, test_size=0.30, random_state=42)
    # Split temp: 50/50 → 15% val, 15% test
    val, test = train_test_split(temp, test_size=0.50, random_state=42)
    
    # Save
    train_path = os.path.join(BASE, f'dataset/{dialect}_train.csv')
    val_path   = os.path.join(BASE, f'dataset/{dialect}_val.csv')
    test_path  = os.path.join(BASE, f'dataset/{dialect}_test.csv')
    
    train.to_csv(train_path, index=False, encoding='utf-8-sig')
    val.to_csv(val_path,     index=False, encoding='utf-8-sig')
    test.to_csv(test_path,   index=False, encoding='utf-8-sig')
    
    print(f'✅ {dialect}: {len(df)} verified clips split into:')
    print(f'   Train: {len(train)} clips → {train_path}')
    print(f'   Val:   {len(val)} clips  → {val_path}')
    print(f'   Test:  {len(test)} clips  → {test_path}')
    print()

## Step 3: Save Script to Drive

In [ ]:
script = '''import pandas as pd
from sklearn.model_selection import train_test_split

for dialect in ['dhaka', 'chittagong']:
    df = pd.read_csv(f'dataset/{dialect}_raw.csv')
    df = df[df.verified == 'YES']
    print(f'{dialect}: {len(df)} verified clips')
    train, temp = train_test_split(df, test_size=0.30, random_state=42)
    val, test = train_test_split(temp, test_size=0.50, random_state=42)
    train.to_csv(f'dataset/{dialect}_train.csv', index=False, encoding='utf-8-sig')
    val.to_csv(f'dataset/{dialect}_val.csv', index=False, encoding='utf-8-sig')
    test.to_csv(f'dataset/{dialect}_test.csv', index=False, encoding='utf-8-sig')
    print(f'  Train:{len(train)} Val:{len(val)} Test:{len(test)}')
'''
with open(os.path.join(BASE, 'scripts/split_dataset.py'), 'w') as f:
    f.write(script)
print('✅ Script saved to scripts/split_dataset.py')

---
## ✅ Part 4 Complete!

**Saved in Google Drive:**
- `dataset/dhaka_train.csv`, `dhaka_val.csv`, `dhaka_test.csv`
- `dataset/chittagong_train.csv`, `chittagong_val.csv`, `chittagong_test.csv`

**Next → Open `Phase1_Part5_Train_Whisper.ipynb`** (the big one!)

---